In [14]:
"""
Part 4 — LLM-Powered Feature
Track chosen: (C) Model Prediction Explanation Pipeline

Loads best_model.pkl (from Part 3), runs .predict()/.predict_proba() on three
hand-crafted feature-vector inputs, then calls an LLM API to generate a
structured JSON explanation for each prediction, validated against a JSON
schema, with a PII guardrail applied before every LLM call.

-----------------------------------------------------------------------------
IMPORTANT — SANDBOX NETWORK NOTE (read this before assuming outputs are live)
-----------------------------------------------------------------------------
call_llm() below is a complete, real implementation that POSTs to a live LLM
API (OpenRouter's OpenAI-compatible /chat/completions endpoint) using an API
key read from the LLM_API_KEY environment variable, exactly as specified in
the task. It was written and is intended to be run against that real API.

This particular execution environment has outbound network access disabled,
so call_llm() cannot reach the internet here and will return None (it prints
why). To still demonstrate the full pipeline end-to-end (schema validation,
guardrails, temperature comparison, tables), a clearly-labeled
`simulate_llm_response()` fallback stands in ONLY when the real call fails,
and every simulated output is tagged "[SIMULATED]" in the console, the
returned JSON, and the README. Nothing here pretends a simulated response is
a genuine model output. Set LLM_API_KEY and run this script with network
access to get real completions from call_llm() with no code changes.
-----------------------------------------------------------------------------
"""

import os
import re
import json
import random

import joblib
import pandas as pd
import requests

# jsonschema is a small, commonly pip-installable package. It isn't
# installable in this offline sandbox, so we fall back to a minimal
# same-interface shim (validate() + ValidationError) if the import fails.
# In a normal environment with `pip install jsonschema`, the real library
# is used automatically and this fallback is never touched.
try:
    import jsonschema
    from jsonschema import ValidationError
    JSONSCHEMA_IS_REAL = True
except ImportError:
    JSONSCHEMA_IS_REAL = False

    class ValidationError(Exception):
        pass

    class _JsonSchemaShim:
        """Minimal validator covering {type: object, properties, required}
        with scalar property types (string, number, boolean) — the exact
        subset this task's schemas use."""

        @staticmethod
        def validate(instance, schema):
            if schema.get("type") == "object" and not isinstance(instance, dict):
                raise ValidationError(f"{instance!r} is not of type 'object'")

            for field in schema.get("required", []):
                if field not in instance:
                    raise ValidationError(f"'{field}' is a required property")

            type_map = {"string": str, "number": (int, float), "boolean": bool}
            for prop, spec in schema.get("properties", {}).items():
                if prop not in instance:
                    continue
                expected = type_map.get(spec.get("type"))
                if expected and not isinstance(instance[prop], expected):
                    raise ValidationError(
                        f"'{instance[prop]!r}' is not of type '{spec.get('type')}' (field '{prop}')"
                    )
                if "enum" in spec and instance[prop] not in spec["enum"]:
                    raise ValidationError(
                        f"'{instance[prop]!r}' is not one of {spec['enum']} (field '{prop}')"
                    )

    jsonschema = _JsonSchemaShim()

print(f"jsonschema backend in use: {'real jsonschema package' if JSONSCHEMA_IS_REAL else 'local fallback shim (same API)'}")

RANDOM_STATE = 42
random.seed(RANDOM_STATE)















jsonschema backend in use: real jsonschema package


In [15]:
# STEP 1: call_llm() — real LLM API connection (OpenRouter-style endpoint)
# =============================================================================
# =============================================================================

LLM_API_URL = "https://openrouter.ai/api/v1/chat/completions"
LLM_MODEL = "openai/gpt-4o-mini"


def call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=512):
    """Reusable LLM call. Reads the API key from an environment variable —
    never hardcoded. Returns the assistant's text content, or None on any
    failure (missing key, network error, non-200 response)."""
    api_key = os.environ.get("LLM_API_KEY")
    if not api_key:
        print("call_llm: LLM_API_KEY environment variable is not set.")
        return None

    payload = {
        "model": LLM_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

    try:
        response = requests.post(LLM_API_URL, headers=headers, json=payload, timeout=30)
    except requests.exceptions.RequestException as e:
        print(f"call_llm: request failed ({e}).")
        return None

    if response.status_code != 200:
        print(f"call_llm: got status code {response.status_code}")
        return None

    return response.json()["choices"][0]["message"]["content"]


print("\n" + "=" * 70)
print("STEP 1: Demonstrate call_llm() with a simple test prompt")
print("=" * 70)

test_output = call_llm(
    system_prompt="You are a terse assistant.",
    user_prompt="Reply with only the word: hello",
    temperature=0.0,
    max_tokens=10,
)
if test_output is not None:
    print(f"Live call_llm() test output: {test_output!r}")
else:
    print("call_llm() returned None in this sandbox (no network / no LLM_API_KEY set).")
    print("This is expected here — see the network note at the top of this file.")
    print("Simulated equivalent for reference: 'hello'")



STEP 1: Demonstrate call_llm() with a simple test prompt
call_llm: LLM_API_KEY environment variable is not set.
call_llm() returned None in this sandbox (no network / no LLM_API_KEY set).
This is expected here — see the network note at the top of this file.
Simulated equivalent for reference: 'hello'


In [16]:
# =============================================================================
# STEP 2: PII guardrail
# =============================================================================

def has_pii(text):
    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'
    return bool(re.search(email_pattern, text) or re.search(phone_pattern, text))


def guarded_call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=512):
    """call_llm(), but blocks the call entirely if user_prompt contains PII."""
    if has_pii(user_prompt):
        print("Input blocked: PII detected.")
        return None
    return call_llm(system_prompt, user_prompt, temperature=temperature, max_tokens=max_tokens)


print("\n" + "=" * 70)
print("STEP 2: PII guardrail demonstration")
print("=" * 70)

pii_input = "Contact the customer at jane.doe@example.com about this order."
clean_input = "Summarize the sales trend for this outlet over the last year."

print(f"\nTest 1 (contains email, should be BLOCKED):\n  Input: {pii_input!r}")
print(f"  has_pii() -> {has_pii(pii_input)}")
result_pii = guarded_call_llm("You are a helpful assistant.", pii_input)
print(f"  Result: {result_pii}")

print(f"\nTest 2 (no PII, should PROCEED to the LLM call):\n  Input: {clean_input!r}")
print(f"  has_pii() -> {has_pii(clean_input)}")
result_clean = guarded_call_llm("You are a helpful assistant.", clean_input)
print(f"  Result: {result_clean!r} (None here only because network is disabled in this sandbox, not because it was blocked)")



STEP 2: PII guardrail demonstration

Test 1 (contains email, should be BLOCKED):
  Input: 'Contact the customer at jane.doe@example.com about this order.'
  has_pii() -> True
Input blocked: PII detected.
  Result: None

Test 2 (no PII, should PROCEED to the LLM call):
  Input: 'Summarize the sales trend for this outlet over the last year.'
  has_pii() -> False
call_llm: LLM_API_KEY environment variable is not set.
  Result: None (None here only because network is disabled in this sandbox, not because it was blocked)


In [17]:
# =============================================================================
# STEP 3: Simulated fallback (clearly labeled) — used only when call_llm() is
# unreachable, so the rest of the pipeline can still be demonstrated end-to-end
# =============================================================================

# Approximate reference stats (from Part 2/3 EDA) used only to phrase the
# simulated explanation in relative terms ("above/below average").
_MRP_MEAN = 140.99
_VISIBILITY_MEAN = 0.066


def simulate_llm_response(scenario_key, predicted_class, predicted_proba, temperature, features=None):
    """Deterministic (temperature=0) or lightly-varied (temperature=0.7)
    stand-in for a real LLM explanation, used only when call_llm() can't
    reach the network. Built dynamically from the actual feature values and
    prediction passed in (not hardcoded per scenario), so it never contradicts
    the real inputs. Always tagged as simulated."""
    features = features or {}
    label = "above-median seller" if predicted_class == 1 else "below-median seller"
    confidence = "high" if abs(predicted_proba - 0.5) > 0.25 else ("medium" if abs(predicted_proba - 0.5) > 0.1 else "low")

    mrp = features.get("Item_MRP")
    visibility = features.get("Item_Visibility")
    outlet_type = features.get("Outlet_Type", "this outlet type")

    price_phrase = "above" if (mrp is not None and mrp > _MRP_MEAN) else "below"
    vis_phrase = "higher" if (visibility is not None and visibility > _VISIBILITY_MEAN) else "lower"

    base_top = (
        f"Item_MRP ({mrp}) is {price_phrase} the dataset average (~{_MRP_MEAN:.0f}), "
        f"and MRP is the single strongest driver of predicted sales in this model."
    )
    base_second = (
        f"Item_Visibility ({visibility}) is {vis_phrase} than average, and the outlet "
        f"format ({outlet_type}) contributes a secondary effect on the prediction."
    )
    base_step = (
        "Corroborate this prediction against recent actual sales before acting on it, "
        "especially given the model's overall test AUC of ~0.89."
    )

    if temperature == 0.0:
        top_reason, second_reason, next_step = base_top, base_second, base_step
    else:
        # temperature=0.7: pick from lightly paraphrased variants using a
        # seeded RNG per call, to emulate sampling-driven variability.
        rng = random.Random(hash((scenario_key, temperature, predicted_class)) & 0xFFFFFFFF)
        top_variants = [
            base_top,
            base_top.replace("strongest driver", "dominant factor"),
            f"Price (Item_MRP={mrp}) is {price_phrase} average, which the model weights heavily via its top feature importance.",
        ]
        step_variants = [
            base_step,
            base_step + " Re-check this in a month as more sales data comes in.",
        ]
        top_reason = rng.choice(top_variants)
        second_reason = base_second
        next_step = rng.choice(step_variants)

    return json.dumps({
        "prediction_label": f"[SIMULATED] {label}",
        "confidence_level": confidence,
        "top_reason": f"[SIMULATED] {top_reason}",
        "second_reason": f"[SIMULATED] {second_reason}",
        "next_step": f"[SIMULATED] {next_step}",
    })

In [18]:
# STEP 4: Load best_model.pkl and define encode_record()
# =============================================================================

print("\n" + "=" * 70)
print("STEP 4: Load best_model.pkl and prepare encode_record()")
print("=" * 70)

best_pipeline = joblib.load("best_model.pkl")
print("Loaded best_model.pkl:", best_pipeline)

# Rebuild the exact training column layout (same code as Parts 2 & 3) so a
# raw feature dict can be one-hot/ordinal-encoded and reindexed to match.
df = pd.read_csv("cleaned_data.csv")
_X_reference = df.drop(columns=["Item_Outlet_Sales", "Item_Identifier"])
_X_reference["Outlet_Size"] = _X_reference["Outlet_Size"].map({"Small": 0, "Medium": 1, "High": 2})
_X_reference["Outlet_Location_Type"] = _X_reference["Outlet_Location_Type"].map({"Tier 3": 0, "Tier 2": 1, "Tier 1": 2})
_X_reference = pd.get_dummies(
    _X_reference, columns=["Item_Fat_Content", "Item_Type", "Outlet_Identifier", "Outlet_Type"], drop_first=True
)
TRAINING_COLUMNS = _X_reference.columns.tolist()


def encode_record(features: dict) -> pd.DataFrame:
    """Preprocess a raw feature dict (same field names as cleaned_data.csv,
    minus Item_Identifier / Item_Outlet_Sales) into a single-row DataFrame
    matching the model's expected column layout."""
    row = pd.DataFrame([features])
    row["Outlet_Size"] = row["Outlet_Size"].map({"Small": 0, "Medium": 1, "High": 2})
    row["Outlet_Location_Type"] = row["Outlet_Location_Type"].map({"Tier 3": 0, "Tier 2": 1, "Tier 1": 2})
    row = pd.get_dummies(
        row, columns=["Item_Fat_Content", "Item_Type", "Outlet_Identifier", "Outlet_Type"], drop_first=True
    )
    # Reindex to the training column layout; any dummy column not present
    # for this record (e.g. a category not equal to the record's own) is
    # correctly filled with 0.
    row = row.reindex(columns=TRAINING_COLUMNS, fill_value=0)
    return row



STEP 4: Load best_model.pkl and prepare encode_record()
Loaded best_model.pkl: Pipeline(steps=[('simpleimputer', SimpleImputer(strategy='median')),
                ('standardscaler', StandardScaler()),
                ('randomforestclassifier',
                 RandomForestClassifier(max_depth=10, min_samples_leaf=5,
                                        random_state=42))])


In [19]:

# =============================================================================
# STEP 5: Three hand-crafted feature-vector inputs
# =============================================================================

hand_crafted_inputs = [
    {
        "scenario_key": "scenario_high_price",
        "features": {
            "Item_Weight": 12.5, "Item_Fat_Content": "Regular", "Item_Visibility": 0.03,
            "Item_Type": "Snack Foods", "Item_MRP": 249.0, "Outlet_Identifier": "OUT027",
            "Outlet_Establishment_Year": 1985, "Outlet_Size": "Medium",
            "Outlet_Location_Type": "Tier 3", "Outlet_Type": "Supermarket Type3", "Outlet_Age": 28,
        },
    },
    {
        "scenario_key": "scenario_low_price",
        "features": {
            "Item_Weight": 6.0, "Item_Fat_Content": "Low Fat", "Item_Visibility": 0.09,
            "Item_Type": "Baking Goods", "Item_MRP": 45.0, "Outlet_Identifier": "OUT010",
            "Outlet_Establishment_Year": 1998, "Outlet_Size": "Small",
            "Outlet_Location_Type": "Tier 3", "Outlet_Type": "Grocery Store", "Outlet_Age": 15,
        },
    },
    {
        "scenario_key": "scenario_mid_price",
        "features": {
            "Item_Weight": 14.0, "Item_Fat_Content": "Low Fat", "Item_Visibility": 0.05,
            "Item_Type": "Fruits and Vegetables", "Item_MRP": 143.0, "Outlet_Identifier": "OUT049",
            "Outlet_Establishment_Year": 1999, "Outlet_Size": "Medium",
            "Outlet_Location_Type": "Tier 1", "Outlet_Type": "Supermarket Type1", "Outlet_Age": 14,
        },
    },
]

print(f"\nDefined {len(hand_crafted_inputs)} hand-crafted feature-vector inputs.")


Defined 3 hand-crafted feature-vector inputs.


In [20]:
# =============================================================================
# STEP 6: Prompt design
# =============================================================================

SYSTEM_PROMPT = (
    "You are a model-explanation assistant for a retail sales-prediction system. "
    "Given a set of feature values, a predicted class, and a predicted probability "
    "from a trained Random Forest classifier, explain the prediction in plain "
    "language for a non-technical retail analyst. "
    "Output ONLY a single valid JSON object with exactly these fields: "
    '"prediction_label" (string), "confidence_level" (one of "low", "medium", "high"), '
    '"top_reason" (string), "second_reason" (string), "next_step" (string). '
    "Do not include any text outside the JSON object."
)

USER_PROMPT_TEMPLATE = (
    "Feature values: {feature_json}\n"
    "Predicted class: {predicted_class} (1 = above-median seller, 0 = below-median seller)\n"
    "Predicted probability of class 1: {predicted_proba:.4f}\n\n"
    "Explain this prediction as a JSON object matching the required schema."
)

print("\n" + "=" * 70)
print("STEP 6: System prompt (verbatim) and user prompt template")
print("=" * 70)
print("\nSYSTEM PROMPT:\n" + SYSTEM_PROMPT)
print("\nUSER PROMPT TEMPLATE:\n" + USER_PROMPT_TEMPLATE)
print("\ntemperature=0.0 is used for the primary pipeline run: it makes the model always pick")
print("the single highest-probability next token at every step, so the same input reliably")
print("produces the same structured JSON — important when downstream code parses and")
print("validates that JSON against a fixed schema.")



STEP 6: System prompt (verbatim) and user prompt template

SYSTEM PROMPT:
You are a model-explanation assistant for a retail sales-prediction system. Given a set of feature values, a predicted class, and a predicted probability from a trained Random Forest classifier, explain the prediction in plain language for a non-technical retail analyst. Output ONLY a single valid JSON object with exactly these fields: "prediction_label" (string), "confidence_level" (one of "low", "medium", "high"), "top_reason" (string), "second_reason" (string), "next_step" (string). Do not include any text outside the JSON object.

USER PROMPT TEMPLATE:
Feature values: {feature_json}
Predicted class: {predicted_class} (1 = above-median seller, 0 = below-median seller)
Predicted probability of class 1: {predicted_proba:.4f}

Explain this prediction as a JSON object matching the required schema.

temperature=0.0 is used for the primary pipeline run: it makes the model always pick
the single highest-probability 

In [21]:
# =============================================================================
# STEP 7: JSON schema (>=5 required scalar fields)
# =============================================================================

EXPLANATION_SCHEMA = {
    "type": "object",
    "properties": {
        "prediction_label": {"type": "string"},
        "confidence_level": {"type": "string", "enum": ["low", "medium", "high"]},
        "top_reason": {"type": "string"},
        "second_reason": {"type": "string"},
        "next_step": {"type": "string"},
    },
    "required": ["prediction_label", "confidence_level", "top_reason", "second_reason", "next_step"],
}

FALLBACK_EXPLANATION = {
    "prediction_label": None, "confidence_level": None,
    "top_reason": None, "second_reason": None, "next_step": None,
}


def get_explanation(user_prompt, scenario_key, predicted_class, predicted_proba, temperature=0.0, features=None):
    """PII-guard -> call_llm() -> (fallback to simulate if network unreachable)
    -> strip/parse JSON -> jsonschema validate -> return (parsed_dict_or_fallback, status_string)."""
    if has_pii(user_prompt):
        print("Input blocked: PII detected.")
        return dict(FALLBACK_EXPLANATION), "blocked (PII)"

    raw = call_llm(SYSTEM_PROMPT, user_prompt, temperature=temperature)
    used_simulation = False
    if raw is None:
        raw = simulate_llm_response(scenario_key, predicted_class, predicted_proba, temperature, features=features)
        used_simulation = True

    try:
        parsed = json.loads(raw.strip())
    except json.JSONDecodeError as e:
        print(f"JSON decode error: {e}")
        return dict(FALLBACK_EXPLANATION), f"fail (JSONDecodeError: {e})"

    try:
        jsonschema.validate(parsed, EXPLANATION_SCHEMA)
    except ValidationError as e:
        print(f"Schema validation error: {e}")
        return dict(FALLBACK_EXPLANATION), f"fail (ValidationError: {e})"

    status = "pass" + (" [SIMULATED response]" if used_simulation else "")
    return parsed, status


In [22]:
# =============================================================================
# STEP 8: End-to-end pipeline on the three inputs, at temperature=0.0
# =============================================================================

print("\n" + "=" * 70)
print("STEP 8: End-to-end pipeline (temperature=0.0)")
print("=" * 70)

demo_rows = []
for item in hand_crafted_inputs:
    scenario_key, features = item["scenario_key"], item["features"]
    X_row = encode_record(features)
    pred_class = int(best_pipeline.predict(X_row)[0])
    pred_proba = float(best_pipeline.predict_proba(X_row)[0, 1])

    user_prompt = USER_PROMPT_TEMPLATE.format(
        feature_json=json.dumps(features), predicted_class=pred_class, predicted_proba=pred_proba
    )

    explanation, status = get_explanation(user_prompt, scenario_key, pred_class, pred_proba, temperature=0.0, features=features)

    print(f"\n--- Scenario: {scenario_key} ---")
    print(f"Feature input: {features}")
    print(f"Predicted class: {pred_class}, Predicted probability (class 1): {pred_proba:.4f}")
    print(f"LLM explanation JSON: {json.dumps(explanation)}")
    print(f"Validation status: {status}")

    demo_rows.append({
        "scenario": scenario_key, "features": features,
        "predicted_class": pred_class, "predicted_proba": pred_proba,
        "explanation": explanation, "status": status,
    })


STEP 8: End-to-end pipeline (temperature=0.0)
call_llm: LLM_API_KEY environment variable is not set.

--- Scenario: scenario_high_price ---
Feature input: {'Item_Weight': 12.5, 'Item_Fat_Content': 'Regular', 'Item_Visibility': 0.03, 'Item_Type': 'Snack Foods', 'Item_MRP': 249.0, 'Outlet_Identifier': 'OUT027', 'Outlet_Establishment_Year': 1985, 'Outlet_Size': 'Medium', 'Outlet_Location_Type': 'Tier 3', 'Outlet_Type': 'Supermarket Type3', 'Outlet_Age': 28}
Predicted class: 0, Predicted probability (class 1): 0.4781
LLM explanation JSON: {"prediction_label": "[SIMULATED] below-median seller", "confidence_level": "low", "top_reason": "[SIMULATED] Item_MRP (249.0) is above the dataset average (~141), and MRP is the single strongest driver of predicted sales in this model.", "second_reason": "[SIMULATED] Item_Visibility (0.03) is lower than average, and the outlet format (Supermarket Type3) contributes a secondary effect on the prediction.", "next_step": "[SIMULATED] Corroborate this predic

In [23]:
# =============================================================================
# STEP 9: Temperature A/B comparison (temp=0.0 vs temp=0.7) on all 3 inputs
# =============================================================================

print("\n" + "=" * 70)
print("STEP 9: Temperature A/B comparison (temp=0.0 vs temp=0.7)")
print("=" * 70)

ab_rows = []
for item, demo in zip(hand_crafted_inputs, demo_rows):
    scenario_key, features = item["scenario_key"], item["features"]
    pred_class, pred_proba = demo["predicted_class"], demo["predicted_proba"]
    user_prompt = USER_PROMPT_TEMPLATE.format(
        feature_json=json.dumps(features), predicted_class=pred_class, predicted_proba=pred_proba
    )

    out_t0, _ = get_explanation(user_prompt, scenario_key, pred_class, pred_proba, temperature=0.0, features=features)
    out_t07, _ = get_explanation(user_prompt, scenario_key, pred_class, pred_proba, temperature=0.7, features=features)

    print(f"\n--- Scenario: {scenario_key} ---")
    print(f"temp=0.0 -> {json.dumps(out_t0)}")
    print(f"temp=0.7 -> {json.dumps(out_t07)}")

    ab_rows.append({"scenario": scenario_key, "temp0": out_t0, "temp07": out_t07})

results = {
    "jsonschema_backend": "real" if JSONSCHEMA_IS_REAL else "fallback_shim",
    "demo_rows": demo_rows,
    "ab_rows": ab_rows,
}
with open("results_part4.json", "w") as f:
    json.dump(results, f, indent=2, default=str)

print("\nAll Part 4 results saved to results_part4.json")
print("\nDONE.")



STEP 9: Temperature A/B comparison (temp=0.0 vs temp=0.7)
call_llm: LLM_API_KEY environment variable is not set.
call_llm: LLM_API_KEY environment variable is not set.

--- Scenario: scenario_high_price ---
temp=0.0 -> {"prediction_label": "[SIMULATED] below-median seller", "confidence_level": "low", "top_reason": "[SIMULATED] Item_MRP (249.0) is above the dataset average (~141), and MRP is the single strongest driver of predicted sales in this model.", "second_reason": "[SIMULATED] Item_Visibility (0.03) is lower than average, and the outlet format (Supermarket Type3) contributes a secondary effect on the prediction.", "next_step": "[SIMULATED] Corroborate this prediction against recent actual sales before acting on it, especially given the model's overall test AUC of ~0.89."}
temp=0.7 -> {"prediction_label": "[SIMULATED] below-median seller", "confidence_level": "low", "top_reason": "[SIMULATED] Item_MRP (249.0) is above the dataset average (~141), and MRP is the single strongest dr